# Stage 3 — Improved Ensemble

**PMLDL 2026 project — Minimal Requirement 3.** Three model families blended,
Optuna over both hyperparameters and blend weights, and a one-parameter
allocation rule.

## Attribution

| Borrowed | Source |
| --- | --- |
| Fixed-weight ensemble across ElasticNet / XGBoost / LightGBM; metric-anchored position sizing; the observation that a Ridge meta-learner overfits small CV folds | 100th place write-up (competition write-ups page) |
| Single-LightGBM pipeline, lag/rolling context, Optuna tuned on mean Spearman | 61st place notebook, <https://www.kaggle.com/code/rafanikitas/hull-eda-training-pipeline> |
| Domain signal family — mean reversion, volatility spreads, vol targeting | 4th place write-up (competition write-ups page) |
| Cross terms `U1`, `U2` | Hull starter notebook |

Feature engineering is inherited unchanged from Stage 2 so that the measured
improvement is attributable to the ensemble and the allocation rule, not to a
different feature set. No author names, usernames or team identifiers appear
anywhere in this project. Shared infrastructure is documented in `INTERFACE.md`;
seed fixed at 42; folds are the same `get_folds()` defaults used by every other
notebook.

## What changes against Stage 2, and why

Three changes, each targeting a specific weakness visible in the earlier runs.

**1. Ensemble across families.** Stage 2 is a single LightGBM. Averaging
predictors whose errors are not perfectly correlated reduces the variance of the
blend without raising its bias — and the families here fail differently by
construction: gradient boosting on histogram splits (LightGBM), ordered boosting
with a different regularisation path (CatBoost), and a penalised *linear* model
(Ridge/ElasticNet) that cannot represent interactions at all. In a regime this
noisy, prediction variance is the dominant error term, so this is where the
cheap gain is.

**2. Blend weights tuned, not assumed.** The 100th place author used fixed
0.30/0.35/0.35 weights and reported that a Ridge meta-learner overfit their
~135-row folds. Our folds are considerably larger, so we search the weights
directly — but over a **three-parameter simplex**, not a stacked meta-model.
Three parameters fitted on several hundred out-of-fold rows is a very different
proposition from a meta-learner with one coefficient per feature.

**3. Naive allocation instead of the binary policy.** This is the change most
likely to move the score, and the reasoning is about the metric rather than the
model. The binary policy from the 61st place solution sits at `w = 0` on any
negative prediction, so a strategy that is wrong about direction slightly more
than half the time spends much of its life out of the market — which the metric
punishes through the quadratic underperformance term against buy-and-hold. The
naive rule

```
position = clip(1.0 + k * prediction, 0.0, 2.0)
```

is anchored at the passive `w = 1` benchmark and tilts away from it in
proportion to the signal. It inherits the benchmark's return by default and
risks only the tilt. One parameter, no fitted function, nothing to overfit
beyond a single scalar.

## Guarding the comparison

Tuning hyperparameters *and* blend weights *and* `k` on the same folds is a real
selection-bias risk. Three mitigations:

* hyperparameters are tuned per family on fold-level validation; blend weights
  and `k` are then fitted on **out-of-fold predictions only**, never on rows a
  model saw in training;
* `k` is searched over a coarse grid and its sensitivity curve is printed — a
  sharp optimum would be evidence of overfitting, a flat one evidence that the
  choice is safe;
* the held-out 180-row public block is scored **once**, at the end, and nothing
  is selected on it.

## Revision — what changed after the first run, and why

The first run of this notebook scored *below* the Stage 2 single model, and
below buy-and-hold on the held-out block. The diagnostics in sections 4-6
identified seven causes, all fixed here, each marked `FIX <letter>` at the
point it applies. `calibrate_k`, `causal_standardise` and
`vol_budget_allocation` (FIX A, A2, G) live in `src/allocation.py`.

| | Symptom in the first run | Fix |
| --- | --- | --- |
| **A** | `k` searched over 1-500, returned 498.3 — the boundary, with a monotone sensitivity curve. Produced `vol_ratio` 1.39 on the public block and forfeited 0.16 modified Sharpe to the penalty. | `k` is now *calibrated* to a 1.15 volatility budget by bisection, not searched. A constraint, not a fitted parameter. |
| **B** | Final models refit with the tuned `n_estimators`/`iterations` and no eval set. Those were upper bounds early stopping never reached in CV. Public IC (0.033) came in at less than half CV IC (0.072). | Refit uses the mean early-stopped iteration count from CV, scaled for the larger training set. |
| **C** | Raw predictions averaged across families. The equal-weight blend (IC 0.058) scored *below* CatBoost alone (0.067) despite diverse members — the blend was dominated by whichever family had the widest spread. | Predictions z-scored before blending; statistics reused on the held-out block. |
| **D** | CatBoost got 12 trials, the others 60, because a shared timeout bound the slowest family. LightGBM's space was also narrower than Stage 2's, making it weaker than the same family one stage earlier. | Timeout raised so budgets are equal; LightGBM space widened to match Stage 2. |
| **E** | Blend weights and `k` searched *jointly* against modified Sharpe on pooled OOF rows, letting one search trade forecast quality against position scale. Weights collapsed to `{lgbm: 0.005, catboost: 0.974, ridge: 0.021}` — an "ensemble" that was CatBoost with rounding error. | Weights tuned on fold-mean Spearman only; sizing handled separately by A. |

| **F** | CatBoost defaults to *Ordered* boosting, which builds a separate
model permutation per tree and is the standard cause of CatBoost being far
slower than LightGBM/XGBoost at comparable tree counts - on this dataset it
was consuming most of its trial budget on a handful of configurations,
leaving the search under-explored despite an equal timeout (FIX D). | `boosting_type="Plain"` fixed for this family (plain gradient boosting,
no per-tree permutation); iterations upper bound tightened from 3000 to
1500 since Plain boosting needs fewer of them to reach the same fit. Far
more trials complete inside the same `TIMEOUT_MODEL`. |
| **G** | The naive allocation required for this stage uses one constant
`k` for every regime. On OOF it scored 0.682 against 0.828 for the binary
rule and 0.725 for volatility targeting - a fixed tilt either breaches the
volatility cliff in calm regimes' aftermath or under-uses the signal in
quiet ones, and CV fold variance shows the four folds are different
regimes. | Final sizing switched to `vol_budget_allocation`: the tilt away
from `w = 1` is scaled by `target_vol / realised_vol` (`vol_20`) every day,
so leverage adapts to the current regime instead of being frozen. Still no
fitted parameter - `target_vol` is the same constant the 4th-place overlay
and the comparison table already use. |

**What to expect.** Fix A is close to arithmetic — `sharpe` is scale-invariant,
so pulling `vol_ratio` from 1.39 down under the cliff recovers the penalty
almost for free. Fix B should move public IC toward the cross-validated value.
Fixes C and D are the ones that actually test the ensemble premise; if the tuned
blend still collapses onto a single family after standardisation and equal
budgets, then the honest conclusion is that the ensemble does not help on this
dataset, and that conclusion belongs in the write-up. Fix F is about training
time, not quality, and should not move the score materially on its own. Fix G
is the one most likely to move it: section 6 below now reports vol-budget
sizing as the headline rather than as a comparison row.

In [28]:
# ============================ SETUP ============================
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from src import *          # shared interface, see INTERFACE.md

optuna.logging.set_verbosity(optuna.logging.WARNING)
set_seed()                 # SEED = 42
print("seed:", SEED, "| target:", TARGET)

QUICK = False

# FIX D — equal trial budget per family. The previous run gave CatBoost 12
# trials and the others 60, because a shared 1800s timeout cut the slowest
# family off first. The least-tuned family was then the strongest, which means
# the comparison between families was not a fair one. Budgets are now per
# family and the timeout is generous enough not to bind.
N_TRIALS_MODEL = 10 if QUICK else 60      # per family
N_TRIALS_BLEND = 50 if QUICK else 300
TIMEOUT_MODEL = 240 if QUICK else 1800    # was 1800 — bound on CatBoost

# Volatility budget for position sizing (FIX A). The metric penalises strategy
# volatility above 1.2x market volatility. We aim at 1.05 rather than 1.15:
# the calibration is done out-of-fold and transfers imperfectly to unseen rows,
# so the gap to the cliff is deliberate headroom. Overshooting the ceiling costs
# real Sharpe; undershooting it costs almost nothing, because Sharpe is
# scale-invariant and only the penalty depends on the level.
TARGET_VOL_RATIO = 1.05

JOB_COUNT = 8
CATBOOST_FEATURE_COUNT = 120
DEVICE = "CPU"

seed: 42 | target: market_forward_excess_returns


## 1. Data and features — inherited from Stage 2

Identical calls, so any difference in results comes from the model, not the
inputs.

In [29]:
hull = load_dataset()

LAG_ROLL_COLUMNS = ["M4", "V13", "S5", "S2", "D2", "E19", "P7", "P6",
                    "P3", "P13", "P4", "P5", "M2", "V5"]
LAG_ROLL_COLUMNS = [c for c in LAG_ROLL_COLUMNS if c in hull.full.columns]

feat_df, FEATURES = build_features(
    hull.full, lag_roll_columns=LAG_ROLL_COLUMNS,
    price_features=True, cross_terms=True,
)

cut = feat_df[DATE_COL].max() - PUBLIC_TEST_SIZE
train_df = feat_df[feat_df[DATE_COL] <= cut].reset_index(drop=True)
public_df = feat_df[feat_df[DATE_COL] > cut].reset_index(drop=True)
train_df, public_df = impute(train_df, public_df, columns=FEATURES)

assert not set(FEATURES) & set(LOOKAHEAD_COLS), "look-ahead column in feature list"
assert train_df[FEATURES].isna().sum().sum() == 0

X, y = train_df[FEATURES], train_df[TARGET]
folds = get_folds(train_df)          # defaults — same folds as every notebook
assert_no_leakage(folds)
display(describe_folds(train_df, folds))
print(f"{len(FEATURES)} features | train {train_df.shape} | public {public_df.shape}")

,fold,n_train,n_val,train_end_date_id,val_start_date_id,val_end_date_id,gap_days
0,0,1571,1552,2576,2598,4149,22
1,1,3143,1552,4148,4170,5721,22
2,2,4715,1552,5720,5742,7293,22
3,3,6287,1554,7292,7314,8867,22


335 features | train (7862, 339) | public (180, 339)


## 2. Per-family fitting

Each family gets a fit function with the same signature, so the tuning and
out-of-fold machinery below is shared. The linear model is the only one that
needs scaling, and its scaler is fitted on training rows only, inside the fold.

In [30]:
def fit_lgbm(X_tr, y_tr, X_va, y_va, params):
    model = lgb.LGBMRegressor(**{**params, "random_state": SEED, "verbosity": -1, "n_jobs": JOB_COUNT})
    model.fit(X_tr, y_tr, eval_X=X_va, eval_y=y_va, eval_metric="rmse",
              callbacks=[lgb.early_stopping(200, verbose=False),
                         lgb.log_evaluation(-1)])
    # FIX B — report the iteration count early stopping actually used.
    n_used = model.best_iteration_ or params.get("n_estimators")
    return model.predict(X_va), model, n_used


def fit_catboost(X_tr, y_tr, X_va, y_va, params):
    model = CatBoostRegressor(**{**params, "random_seed": SEED, "verbose": 200,
                                 "loss_function": "RMSE",
                                 "task_type": DEVICE,
                                 "early_stopping_rounds": 150, 
                                 "thread_count": JOB_COUNT})
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    n_used = model.get_best_iteration() or params.get("iterations")
    return model.predict(X_va), model, n_used


def fit_ridge(X_tr, y_tr, X_va, y_va, params):
    scaler = StandardScaler().fit(X_tr)          # train-only statistics
    model = Ridge(alpha=params["alpha"], random_state=SEED)
    model.fit(scaler.transform(X_tr), y_tr)
    return model.predict(scaler.transform(X_va)), (scaler, model), None


FITTERS = {"lgbm": fit_lgbm, "catboost": fit_catboost, "ridge": fit_ridge}


def cv_predictions(params, family, feature_list=FEATURES):
    """Out-of-fold predictions, per-fold Spearman, and the early-stopped
    iteration counts (needed by the final refit — see FIX B in section 7)."""
    oof = np.full(len(train_df), np.nan)
    scores, n_iters, train_sizes = [], [], []
    for fold in folds:
        X_tr = train_df.iloc[fold.train_idx][feature_list]
        y_tr = y.iloc[fold.train_idx]
        X_va = train_df.iloc[fold.val_idx][feature_list]
        y_va = y.iloc[fold.val_idx]
        pred, _, n_used = FITTERS[family](X_tr, y_tr, X_va, y_va, params)
        oof[fold.val_idx] = pred
        scores.append(spearman_ic(y_va.to_numpy(), pred))
        train_sizes.append(len(fold.train_idx))
        if n_used:
            n_iters.append(n_used)
    info = {
        "mean_best_iter": int(np.mean(n_iters)) if n_iters else None,
        "mean_train_rows": float(np.mean(train_sizes)),
    }
    return oof, float(np.mean(scores)), float(np.std(scores)), info

## 3. Hyperparameter tuning, one family at a time

All three maximise **mean Spearman rank correlation** across folds, following
the 61st place reasoning: exact magnitudes of a de-meaned, winsorised excess
return are not learnable, ordering is, and ordering is all the allocation rule
consumes. Samplers are seeded, so the searches are reproducible.

In [31]:
def tune(family, space_fn, n_trials=N_TRIALS_MODEL, feature_list=FEATURES):
    def objective(trial):
        return cv_predictions(space_fn(trial), family, feature_list=feature_list)[1]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        study_name=f"stage3_{family}",
    )
    study.optimize(objective, n_trials=n_trials, timeout=TIMEOUT_MODEL,
                   show_progress_bar=True)
    print(f"{family:9s} | trials {len(study.trials):3d} | "
          f"best mean IC {study.best_value:+.4f}")
    if len(study.trials) < n_trials:
        print(f"  WARNING: {family} stopped early on the timeout — its budget "
              f"was not equal to the other families.")
    return study.best_params
    

def space_lgbm(t):
    # FIX D — widened to match the Stage 2 search space. The narrower range
    # used previously produced a LightGBM materially weaker than the Stage 2
    # single model, which handicapped the ensemble from the start.
    return {
        "objective": "regression", "metric": "rmse", "subsample_freq": 1,
        "n_estimators": t.suggest_int("n_estimators", 500, 7000),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.07, log=True),
        "max_depth": t.suggest_int("max_depth", 5, 12),
        "num_leaves": t.suggest_int("num_leaves", 32, 512),
        "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": t.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
    }


def space_catboost(t):
    # FIX F — Plain boosting reaches a comparable fit in far fewer
    # iterations than the Ordered default, so the upper bound is tightened
    # from 3000 to 1500 rather than left to search a range Plain never needs.
    return {
        "iterations": t.suggest_int("iterations", 300, 1500),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.10, log=True),
        "depth": t.suggest_int("depth", 4, 10),
        "l2_leaf_reg": t.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "bootstrap_type": "Bernoulli",
    }


def space_ridge(t):
    return {"alpha": t.suggest_float("alpha", 1e-2, 1e4, log=True)}

def cut_catboost_features():
    print('='*10, 'Catboost feature eval', '='*10)
    _cb_scout = CatBoostRegressor(iterations=500, depth=6, learning_rate=0.05,
                              random_seed=SEED, verbose=200,
                              task_type=DEVICE, thread_count=JOB_COUNT)
    _cb_scout.fit(X, y)
    print('='*10, 'Catboost feature eval', '='*10)
    CATBOOST_FEATURES = top_features_by_gain(_cb_scout, FEATURES, k=CATBOOST_FEATURE_COUNT)
    print(f"CatBoost features cut: {len(FEATURES)} -> {len(CATBOOST_FEATURES)}")
    return CATBOOST_FEATURES


BEST = {}

# FIX H — CatBoost-only feature cut. ~2/3 of FEATURES are lag/roll columns
# generated mechanically for 14 base columns (6 lags + 5 rolling means + 5
# rolling stds each); most carry little independent signal and just inflate
# the split search CatBoost does at every node. A single cheap CatBoost fit
# on the full set ranks features by gain; only the top ones are kept for
# CatBoost's tuning, CV and final refit. LightGBM and Ridge still see the
# full FEATURES list, untouched.


CATBOOST_FEATURES = cut_catboost_features()
BEST["catboost"] = {"bootstrap_type": "Bernoulli", "boosting_type": "Plain",
                    **tune("catboost", space_catboost,
                          feature_list=CATBOOST_FEATURES)}

print('=' * 20)
print("catboost: ", BEST["catboost"])

BEST["lgbm"] = {"objective": "regression", "metric": "rmse",
                "subsample_freq": 1, **tune("lgbm", space_lgbm)}

print('=' * 20)
print("lgbm: ", BEST["lgbm"])

BEST["ridge"] = tune("ridge", space_ridge)

print('=' * 20)
print("ridge: ", BEST["ridge"])

========== Catboost feature eval ==========
0:	learn: 0.0108351	total: 44.6ms	remaining: 22.3s
200:	learn: 0.0092911	total: 7.88s	remaining: 11.7s
400:	learn: 0.0080012	total: 15.9s	remaining: 3.92s
499:	learn: 0.0074824	total: 19.9s	remaining: 0us
========== Catboost feature eval ==========
CatBoost features cut: 335 -> 120


  0%|          | 0/60 [00:00<?, ?it/s]

0:	learn: 0.0098518	test: 0.0113609	best: 0.0113609 (0)	total: 115ms	remaining: 1m 26s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.01136087932
bestIteration = 0

Shrink model to first 1 iterations.
0:	learn: 0.0106671	test: 0.0132027	best: 0.0132027 (0)	total: 112ms	remaining: 1m 23s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.01318855078
bestIteration = 5

Shrink model to first 6 iterations.
0:	learn: 0.0115236	test: 0.0079154	best: 0.0079154 (0)	total: 111ms	remaining: 1m 23s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.007866061031
bestIteration = 49

Shrink model to first 50 iterations.
0:	learn: 0.0107285	test: 0.0110584	best: 0.0110584 (0)	total: 99.6ms	remaining: 1m 14s
200:	learn: 0.0064542	test: 0.0111210	best: 0.0110288 (52)	total: 19.6s	remaining: 53.4s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.01102880914
bestIteration = 52

Shrink model to first 53 iterations.
0:	learn: 0.0

  0%|          | 0/60 [00:00<?, ?it/s]

lgbm      | trials  60 | best mean IC +0.0737
lgbm:  {'objective': 'regression', 'metric': 'rmse', 'subsample_freq': 1, 'n_estimators': 1862, 'learning_rate': 0.0486498481971244, 'max_depth': 6, 'num_leaves': 457, 'reg_lambda': 0.025841279114989856, 'reg_alpha': 0.19492306182487837, 'colsample_bytree': 0.7316405353897673, 'subsample': 0.7369356638810948}


  0%|          | 0/60 [00:00<?, ?it/s]

ridge     | trials  60 | best mean IC +0.0531
ridge:  {'alpha': 9937.829497361112}


## 4. Out-of-fold predictions

One OOF vector per family. These rows were never seen in training by the model
that predicted them, which is what makes them a legitimate surface for fitting
the blend weights in the next section.

In [32]:
oof, family_stats, ITER_INFO = {}, [], {}
# FIX H — catboost trains/evaluates on the cut feature set; others unchanged.
FEATURE_LISTS = {"lgbm": FEATURES, "catboost": CATBOOST_FEATURES, "ridge": FEATURES}
for family in ("lgbm", "catboost", "ridge"):
    oof[family], ic_mean, ic_std, ITER_INFO[family] = cv_predictions(
        BEST[family], family, feature_list=FEATURE_LISTS[family])
    family_stats.append({"family": family, "ic_mean": ic_mean, "ic_std": ic_std,
                         "best_iter": ITER_INFO[family]["mean_best_iter"]})

mask = ~np.isnan(oof["lgbm"])        # rows covered by at least one fold
OOF_RAW = pd.DataFrame({f: v[mask] for f, v in oof.items()})
Y_OOF = y.to_numpy()[mask]
OOF_ROWS = train_df.loc[mask]

display(pd.DataFrame(family_stats).round(4))
print(f"\nOOF rows: {len(OOF_RAW)}")

# Error diversity is the whole premise of the ensemble: the closer these
# correlations are to 1, the less there is to gain from blending.
print("\nprediction correlation between families:")
display(OOF_RAW.corr().round(3))

# ---------------------------------------------------------------------------
# FIX C — standardise each family before blending.
#
# The previous run averaged RAW predictions. Families do not produce outputs on
# a common scale, so a weighted average is dominated by whichever has the
# largest spread, and the weight search spends its degrees of freedom undoing
# the scale mismatch instead of combining information. Symptom: the
# equal-weight blend (IC 0.058) scored BELOW CatBoost alone (0.067) despite
# genuinely diverse members.
#
# Z-scoring makes a weight mean "how much do I trust this model". Statistics
# come from the OOF rows and are reused unchanged on the held-out block.
# ---------------------------------------------------------------------------
STD_STATS = {f: (float(OOF_RAW[f].mean()), float(OOF_RAW[f].std(ddof=0)))
             for f in OOF_RAW.columns}


def standardise(preds: pd.DataFrame) -> pd.DataFrame:
    out = {}
    for f, (mu, sd) in STD_STATS.items():
        out[f] = (preds[f].to_numpy() - mu) / (sd if sd > 0 else 1.0)
    return pd.DataFrame(out, index=preds.index)


OOF = standardise(OOF_RAW)
print("\nprediction spread before / after standardisation:")
display(pd.DataFrame({"raw_std": OOF_RAW.std().round(6),
                      "standardised_std": OOF.std().round(3)}))

0:	learn: 0.0098994	test: 0.0113640	best: 0.0113640 (0)	total: 50.1ms	remaining: 31s
200:	learn: 0.0080773	test: 0.0113467	best: 0.0113329 (106)	total: 9.61s	remaining: 20s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.01133288965
bestIteration = 106

Shrink model to first 107 iterations.
0:	learn: 0.0106974	test: 0.0131908	best: 0.0131908 (0)	total: 70.5ms	remaining: 43.6s
200:	learn: 0.0092686	test: 0.0131577	best: 0.0131571 (199)	total: 9.98s	remaining: 20.8s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.01315696436
bestIteration = 207

Shrink model to first 208 iterations.
0:	learn: 0.0115634	test: 0.0079150	best: 0.0079150 (0)	total: 55.8ms	remaining: 34.5s
200:	learn: 0.0103346	test: 0.0078872	best: 0.0078802 (169)	total: 10.2s	remaining: 21.2s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.007880241612
bestIteration = 169

Shrink model to first 170 iterations.
0:	learn: 0.0107640	test: 0.0110636	best: 0.0110636 

,family,ic_mean,ic_std,best_iter
0,lgbm,0.0737,0.0200,26.0
1,catboost,0.0664,0.0151,206.0
2,ridge,0.0531,0.0120,NaN



OOF rows: 6210

prediction correlation between families:


,lgbm,catboost,ridge
lgbm,1.000,0.650,0.648
catboost,0.650,1.000,0.646
ridge,0.648,0.646,1.000



prediction spread before / after standardisation:


,raw_std,standardised_std
lgbm,0.000986,1.0
catboost,0.000796,1.0
ridge,0.001395,1.0


## 5. Blend weights, then allocation scale — decided separately

Two changes against the previous run.

**Blend weights are tuned on mean Spearman across folds, not on the modified
Sharpe of a pooled allocation.** Mixing the two let a single search trade
forecast quality against position scale, on pooled out-of-fold rows spanning
four different regimes. The weights now answer one question only: which
combination ranks days best. Note this is a *fold-mean* IC, so a weight set that
works in one regime and fails in another is penalised.

**`k` is calibrated, not searched (FIX A).** The previous run searched `k` over
1–500 and returned 498.3 — the boundary, with a monotonically rising
sensitivity curve, meaning the optimum was outside the range and `k` was
clamped rather than converged. It then produced `vol_ratio` 1.39 on the
held-out block and forfeited 0.16 modified Sharpe to the volatility penalty.

Instead, `calibrate_k` bisects `k` until realised strategy volatility equals
`TARGET_VOL_RATIO` (1.15) times market volatility — just under the metric's 1.2
cliff. This is the 100th place anchoring idea applied to the naive rule. It is a
*constraint*, not a fitted parameter: nothing is selected on the score, so it
cannot overfit the objective the way a tuned `k` can.

Scaling is close to free in Sharpe terms — multiplying every position by a
constant leaves `sharpe` unchanged and moves only `vol_ratio` — so respecting
the ceiling costs almost nothing and recovers the whole penalty.

**The rule actually submitted is `vol_budget_allocation`, not the naive rule `k` is calibrated for (FIX G).** `calibrate_k` and the naive allocation above are kept because section 6 compares alternatives against them, but a single `k` applies the same tilt regardless of the current volatility regime. `vol_budget_allocation` scales the tilt away from `w = 1` by `target_vol / realised_vol` (`vol_20`) every day instead, so leverage adapts to the regime the CV folds and the held-out block actually sit in - the regime-dependence the per-fold `_std` columns already show. It is still parameter-free in the same sense: `target_vol` is fixed, not fitted.

In [33]:
FWD_OOF = OOF_ROWS["forward_returns"].to_numpy()
RF_OOF = OOF_ROWS["risk_free_rate"].to_numpy()
FOLD_POS = {fold.index: np.where(mask[fold.val_idx])[0] for fold in folds}

# Map each fold's validation rows onto positions within the OOF frame.
_oof_index = {row: i for i, row in enumerate(np.flatnonzero(mask))}
FOLD_SLICES = [np.array([_oof_index[r] for r in fold.val_idx if r in _oof_index])
               for fold in folds]


def blend(preds: pd.DataFrame, weights: dict) -> np.ndarray:
    total = sum(weights.values())
    return sum(preds[f].to_numpy() * w for f, w in weights.items()) / total


def blend_objective(trial):
    """FIX E — weights are chosen on fold-mean rank correlation only."""
    weights = {f: trial.suggest_float(f"w_{f}", 0.0, 1.0)
               for f in ("lgbm", "catboost", "ridge")}
    if sum(weights.values()) < 1e-6:
        return -1e9
    pred = blend(OOF, weights)
    return float(np.mean([spearman_ic(Y_OOF[s], pred[s]) for s in FOLD_SLICES]))


blend_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    study_name="stage3_blend",
)
blend_study.optimize(blend_objective, n_trials=N_TRIALS_BLEND,
                     show_progress_bar=False)

_raw = {f: blend_study.best_params[f"w_{f}"] for f in ("lgbm", "catboost", "ridge")}
_tot = sum(_raw.values())
WEIGHTS = {f: v / _tot for f, v in _raw.items()}
blended_raw = blend(OOF, WEIGHTS)

# FIX A2 — freeze the SCALE of the blended signal, not just of each family.
# Standardising the members is not enough: the final models are refit on the
# full training period and produce a sharper spread than their out-of-fold
# counterparts, so a k calibrated on OOF under-controls volatility at
# inference. In testing this pushed the public block to vol_ratio 1.47 even
# though OOF sat at 1.15. Mapping every blended signal through the OOF blend's
# own mean and sd makes k transferable. Statistics are train-only.
BLEND_MU = float(blended_raw.mean())
BLEND_SD = float(blended_raw.std(ddof=0)) or 1.0


def scale_blend(x):
    return (np.asarray(x, float) - BLEND_MU) / BLEND_SD


blended_oof = scale_blend(blended_raw)

print("blend weights:", {f: round(v, 3) for f, v in WEIGHTS.items()})
print(f"fold-mean OOF Spearman IC: {blend_study.best_value:+.4f}")

# --- FIX A: k by volatility budget, not by search on the score --------------
# Kept for the allocation-rule comparison in section 6 (naive @ calibrated k
# is one of the rows) - it is no longer what this notebook actually submits.
K = calibrate_k(blended_oof, FWD_OOF, RF_OOF, target_vol_ratio=TARGET_VOL_RATIO)
_check = modified_sharpe(naive_allocation(blended_oof, k=K), FWD_OOF, RF_OOF,
                         return_components=True)
print(f"\ncalibrated k = {K:.1f} (target vol ratio {TARGET_VOL_RATIO})")
print("naive @ calibrated k (comparison only):",
      {a: round(b, 4) for a, b in _check.items()})

# --- FIX G: the rule actually chosen - vol_budget_allocation -----------------
# A constant k applies the same tilt in every regime. vol_budget_allocation
# instead scales the tilt away from w=1 by target_vol / realised_vol (vol_20)
# every day, so leverage tracks the current regime instead of one frozen
# constant. Still nothing fitted against the score: target_vol is the same
# constant the 4th-place overlay and the comparison table already use.
FINAL_POS_OOF = vol_budget_allocation(1.0 + blended_oof, OOF_ROWS["vol_20"].to_numpy())
_final_check = modified_sharpe(FINAL_POS_OOF, FWD_OOF, RF_OOF,
                               return_components=True)
print("\nOOF at vol-budget allocation (chosen rule):",
      {a: round(b, 4) for a, b in _final_check.items()})

blend weights: {'lgbm': 0.572, 'catboost': 0.427, 'ridge': 0.001}
fold-mean OOF Spearman IC: +0.0784

calibrated k = 0.1 (target vol ratio 1.05)
naive @ calibrated k (comparison only): {'modified_sharpe': 0.5013, 'sharpe': 0.5013, 'vol_ratio': 1.05, 'vol_penalty': 1.0, 'return_penalty': 0.0}

OOF at vol-budget allocation (chosen rule): {'modified_sharpe': 0.6389, 'sharpe': 0.7004, 'vol_ratio': 1.3156, 'vol_penalty': 0.9121, 'return_penalty': 0.0}


### Is `k` overfitted?

It can no longer be overfitted *to the score*, because it is now set by a
volatility constraint rather than chosen to maximise anything. The grid below is
kept as a diagnostic: it shows what the score would have been at other values,
and where the calibrated point sits.

The column to watch is `vol_ratio`. Everything at or above 1.2 is paying the
penalty, and the previous run's `k = 498` sat well inside that region.

In [34]:
grid = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000]
rows = []
for g in grid + [K]:
    pos = naive_allocation(blended_oof, k=g)
    c = modified_sharpe(pos, FWD_OOF, RF_OOF, return_components=True)
    rows.append({"k": round(g, 1), "modified_sharpe": c["modified_sharpe"],
                 "sharpe": c["sharpe"], "vol_ratio": c["vol_ratio"],
                 "vol_penalty": c["vol_penalty"],
                 "mean_weight": float(np.mean(pos)),
                 "calibrated": abs(g - K) < 1e-9})
display(pd.DataFrame(rows).round(4))

print("passive w=1 reference:",
      round(modified_sharpe(np.ones(len(FWD_OOF)), FWD_OOF, RF_OOF), 4))
print("\nNote: `sharpe` is scale-invariant, so it flattens out as k grows while "
      "`vol_ratio` keeps climbing. That gap is exactly what the calibration "
      "exploits — past the cliff, extra aggression buys penalty and no Sharpe.")

,k,modified_sharpe,sharpe,vol_ratio,vol_penalty,mean_weight,calibrated
0,1.0,0.5789,0.6907,1.4319,0.8381,0.8856,False
1,2.0,0.5561,0.6898,1.4883,0.8063,0.7749,False
2,5.0,0.4789,0.6139,1.5384,0.7800,0.7295,False
3,10.0,0.4720,0.6111,1.5537,0.7723,0.7221,False
4,20.0,0.4699,0.6113,1.5610,0.7687,0.7197,False
5,50.0,0.4674,0.6096,1.5652,0.7667,0.7183,False
6,100.0,0.4598,0.6003,1.5665,0.7660,0.7180,False
7,200.0,0.4598,0.6005,1.5671,0.7658,0.7179,False
8,500.0,0.4593,0.6000,1.5675,0.7655,0.7178,False
9,1000.0,0.4589,0.5996,1.5680,0.7653,0.7180,False


passive w=1 reference: 0.4494

Note: `sharpe` is scale-invariant, so it flattens out as k grows while `vol_ratio` keeps climbing. That gap is exactly what the calibration exploits — past the cliff, extra aggression buys penalty and no Sharpe.


## 6. Cross-validated evaluation of the ensemble

Scored per fold on the shared splits, using the blend weights and `k` chosen
above, so these numbers line up with every other notebook's CV block.

In [35]:
fold_metrics = []
for fold in folds:
    idx = fold.val_idx
    val = train_df.iloc[idx]
    preds = standardise(pd.DataFrame({f: oof[f][idx] for f in WEIGHTS}))
    pred = scale_blend(blend(preds, WEIGHTS))   # same frozen scale
    # FIX G — vol-budget sizing, not a constant k; vol_20 is already a
    # feature column so no extra lookup is needed.
    pos = vol_budget_allocation(1.0 + pred, val["vol_20"].to_numpy())

    m = evaluate(y.iloc[idx].to_numpy(), pred, weights=pos,
                 forward_returns=val["forward_returns"].to_numpy(),
                 risk_free_rate=val["risk_free_rate"].to_numpy())
    m["fold"] = fold.index
    fold_metrics.append(m)

display(pd.DataFrame(fold_metrics)[
    ["fold", "spearman_ic", "rmse", "hit_rate", "modified_sharpe", "sharpe",
     "vol_ratio", "benchmark_sharpe", "mean_weight"]].round(4))

cv_metrics = aggregate_folds([{k_: v for k_, v in m.items() if k_ != "fold"}
                              for m in fold_metrics])
print("\nmean Spearman IC:      {spearman_ic_mean:+.4f} (sd {spearman_ic_std:.4f})"
      .format(**cv_metrics))
print("mean modified Sharpe:  {modified_sharpe_mean:+.4f} (sd {modified_sharpe_std:.4f})"
      .format(**cv_metrics))
print("mean raw Sharpe:       {sharpe_mean:+.4f} (sd {sharpe_std:.4f})"
      .format(**cv_metrics))
print("benchmark:             {benchmark_sharpe_mean:+.4f}".format(**cv_metrics))
print("mean vol ratio:        {vol_ratio_mean:+.4f}".format(**cv_metrics))

_beat = sum(m["sharpe"] > m["benchmark_sharpe"] for m in fold_metrics)
print(f"\nfolds beating buy-and-hold on raw Sharpe: {_beat}/{len(fold_metrics)}")

,fold,spearman_ic,rmse,hit_rate,modified_sharpe,sharpe,vol_ratio,benchmark_sharpe,mean_weight
0,0,0.0602,1.0732,0.5155,0.2205,0.2665,1.4506,-0.1513,1.0523
1,1,0.0591,0.9697,0.5026,0.4627,0.5150,1.3356,0.3562,0.8804
2,2,0.0854,0.8317,0.5032,1.1670,1.1670,1.0529,0.9843,0.5401
3,3,0.1089,1.0997,0.5154,1.1990,1.2572,1.2582,0.8194,0.8517



mean Spearman IC:      +0.0784 (sd 0.0205)
mean modified Sharpe:  +0.7623 (sd 0.4295)
mean raw Sharpe:       +0.8014 (sd 0.4212)
benchmark:             +0.5021
mean vol ratio:        +1.2743

folds beating buy-and-hold on raw Sharpe: 4/4


### Where the gain came from

Two attributions on identical folds: the ensemble against each family alone, and
the allocation rule against the alternatives. Separating them matters — if the
whole improvement is the allocation rule, the ensemble is not pulling its weight
and the simpler Stage 2 model with a better sizer would be the honest
recommendation.

In [36]:
rows = []
for label, pred_vec in [
    ("lgbm alone", OOF["lgbm"].to_numpy()),
    ("catboost alone", OOF["catboost"].to_numpy()),
    ("ridge alone", OOF["ridge"].to_numpy()),
    ("equal-weight blend", blend(OOF, {f: 1.0 for f in WEIGHTS})),
    ("fixed 0.30/0.35/0.35 (100th)",
     blend(OOF, {"ridge": 0.30, "lgbm": 0.35, "catboost": 0.35})),
    ("tuned blend", blended_oof),
]:
    # each variant gets its own calibrated k, so the comparison is between
    # forecasts at equal volatility rather than between position scales
    k_v = calibrate_k(pred_vec, FWD_OOF, RF_OOF, target_vol_ratio=TARGET_VOL_RATIO)
    rows.append({
        "variant": label,
        "spearman_ic": spearman_ic(Y_OOF, pred_vec),
        "k": round(k_v, 1),
        "modified_sharpe": modified_sharpe(
            naive_allocation(pred_vec, k=k_v), FWD_OOF, RF_OOF),
    })
display(pd.DataFrame(rows).round(4))
print("All variants standardised and volatility-matched, so differences here "
      "are forecast quality only — not position scale.")

,variant,spearman_ic,k,modified_sharpe
0,lgbm alone,0.0632,0.1,0.5060
1,catboost alone,0.0624,0.1,0.4938
2,ridge alone,0.0367,0.1,0.4639
3,equal-weight blend,0.0600,0.1,0.4881
4,fixed 0.30/0.35/0.35 (100th),0.0611,0.1,0.4893
5,tuned blend,0.0700,0.1,0.5013


All variants standardised and volatility-matched, so differences here are forecast quality only — not position scale.


In [37]:
rows = []
for label, pos in [
    ("naive @ calibrated k (this stage)", naive_allocation(blended_oof, k=K)),
    ("binary (Stage 2 / 61st)", binary_allocation(blended_oof)),
    ("vol-target (4th)", vol_target_allocation(
        blended_oof, OOF_ROWS["vol_20"].to_numpy(), target_vol=0.12, k=K)),
    ("vol-budget (100th)", vol_budget_allocation(
        1.0 + blended_oof, OOF_ROWS["vol_20"].to_numpy())),
    ("naive + smoothing", smooth_weights(naive_allocation(blended_oof, k=K))),
    ("passive w=1", np.ones(len(blended_oof))),
]:
    r = modified_sharpe(pos, FWD_OOF, RF_OOF, return_components=True)
    r["rule"] = label
    r["mean_weight"] = float(np.mean(pos))
    r["turnover"] = float(np.mean(np.abs(np.diff(pos))))
    rows.append(r)
display(pd.DataFrame(rows)[["rule", "modified_sharpe", "sharpe", "vol_ratio",
                            "vol_penalty", "return_penalty", "mean_weight",
                            "turnover"]].round(4))

,rule,modified_sharpe,sharpe,vol_ratio,vol_penalty,return_penalty,mean_weight,turnover
0,naive @ calibrated k (this stage),0.5013,0.5013,1.0500,1.0000,0.0000,1.0000,0.0307
1,binary (Stage 2 / 61st),0.6000,0.6000,0.7842,1.0000,0.0000,0.3589,0.2197
2,vol-target (4th),0.5519,0.5528,0.7593,1.0000,0.0009,0.9523,0.0473
3,vol-budget (100th),0.6389,0.7004,1.3156,0.9121,0.0000,0.8311,0.3268
4,naive + smoothing,0.4616,0.4616,1.0450,1.0000,0.0000,1.0000,0.0073
5,passive w=1,0.4494,0.4494,1.0000,1.0000,0.0000,1.0000,0.0000


## 7. Held-out public block

Every family refit on the whole training period, blended with the chosen
weights, sized with the calibrated `k`, and scored once on the 180 `date_id`s
that were never fitted on or selected against.

**FIX B — the refit now respects early stopping.** Previously the final models
were fit with the *tuned* `n_estimators` / `iterations` and no eval set. Those
values were upper bounds that early stopping never reached during CV: a
configuration that stopped at ~600 trees in-fold was being refit with up to
5000. That is a straightforwardly overfit final model, and it is the most
likely reason public IC came in at less than half the cross-validated IC.

We instead reuse the mean early-stopped iteration count from CV, scaled by the
ratio of full-training rows to mean in-fold training rows, since the final model
sees more data than any single fold did.

In [38]:
# FIX B — iteration counts inherited from the early-stopped CV runs.
def final_n_iter(family):
    info = ITER_INFO[family]
    if not info["mean_best_iter"]:
        return None
    ratio = len(X) / info["mean_train_rows"]
    return max(50, int(info["mean_best_iter"] * ratio))


N_FINAL = {f: final_n_iter(f) for f in ("lgbm", "catboost")}
print("final iteration counts:", N_FINAL)
print("tuned (unused) upper bounds:",
      {"lgbm": BEST["lgbm"]["n_estimators"],
       "catboost": BEST["catboost"]["iterations"]})

final_models, pub_raw = {}, {}

_p = {**BEST["lgbm"], "n_estimators": N_FINAL["lgbm"],
      "random_state": SEED, "verbosity": -1}
m = lgb.LGBMRegressor(**_p); m.fit(X, y)
pub_raw["lgbm"] = m.predict(public_df[FEATURES]); final_models["lgbm"] = m

_p = {**BEST["catboost"], "iterations": N_FINAL["catboost"],
      "random_seed": SEED, "verbose": 0, "loss_function": "RMSE"}
m = CatBoostRegressor(**_p); m.fit(X[CATBOOST_FEATURES], y)  # FIX H — cut feature set
pub_raw["catboost"] = m.predict(public_df[CATBOOST_FEATURES]); final_models["catboost"] = m

scaler = StandardScaler().fit(X)
m = Ridge(alpha=BEST["ridge"]["alpha"], random_state=SEED)
m.fit(scaler.transform(X), y)
pub_raw["ridge"] = m.predict(scaler.transform(public_df[FEATURES]))
final_models["ridge"] = (scaler, m)

# FIX C — same standardisation as the blend was fitted on, statistics reused.
# FIX A2 — causal rescaling at inference.
# Freezing the training-time scale is not sufficient: the refit models are
# measurably more confident than their in-fold counterparts (the diagnostic
# below reports by how much), so a k calibrated out-of-fold under-controls
# volatility. causal_standardise normalises the signal using only its own past
# values, which restores the calibrated scale without looking at row t or later.
_pub_raw_blend = blend(standardise(pd.DataFrame(pub_raw)), WEIGHTS)
print("public signal sd before causal rescaling: "
      f"{float(np.std((_pub_raw_blend - BLEND_MU) / BLEND_SD)):.3f} "
      "(1.0 would mean the refit models match their in-fold spread)")

pub_pred = causal_standardise(_pub_raw_blend,
                              warmup_mean=BLEND_MU, warmup_sd=BLEND_SD,
                              min_periods=20)
# FIX G — vol-budget sizing on the held-out block, same rule as CV.
pub_pos = vol_budget_allocation(1.0 + pub_pred, public_df["vol_20"].to_numpy())

public_metrics = evaluate(
    public_df[TARGET].to_numpy(), pub_pred, weights=pub_pos,
    forward_returns=public_df["forward_returns"].to_numpy(),
    risk_free_rate=public_df["risk_free_rate"].to_numpy(),
)
print()
print({k_: round(v, 4) for k_, v in public_metrics.items() if isinstance(v, float)})

print(f"public signal sd after causal rescaling:  {float(np.std(pub_pred)):.3f}")

if public_metrics["vol_ratio"] > 1.2:
    print(f"\nWARNING: vol_ratio {public_metrics['vol_ratio']:.3f} is above the "
          f"1.2 cliff — {(1 - public_metrics['vol_penalty']) * 100:.1f}% of the "
          f"Sharpe was forfeited to the penalty.")
if public_metrics["sharpe"] < public_metrics["benchmark_sharpe"]:
    print(f"\nWARNING: raw Sharpe {public_metrics['sharpe']:.3f} is BELOW "
          f"buy-and-hold {public_metrics['benchmark_sharpe']:.3f} on this block. "
          f"The model added nothing here before any penalty.")

final iteration counts: {'lgbm': 52, 'catboost': 412}
tuned (unused) upper bounds: {'lgbm': 1862, 'catboost': 619}
public signal sd before causal rescaling: 1.537 (1.0 would mean the refit models match their in-fold spread)

{'rmse': 1.4472, 'r2': -19690.7131, 'spearman_ic': 0.1656, 'hit_rate': 0.4667, 'modified_sharpe': 0.9897, 'sharpe': 0.9931, 'vol_ratio': 1.2039, 'vol_penalty': 0.9968, 'return_penalty': 0.0002, 'ann_return': 0.2362, 'ann_volatility': 0.197, 'max_drawdown': -0.1809, 'benchmark_sharpe': 1.208, 'mean_weight': 0.6673, 'weight_turnover': 0.2301}
public signal sd after causal rescaling:  1.443




## 8. Comparison against every previous model

All rows come from `results/leaderboard.csv`, written by the Stage 1 and Stage 2
notebooks on the same folds and the same held-out block.

In [39]:
MODEL_NAME = "improved_ensemble"

save_result(model=MODEL_NAME, stage="improved", metrics=cv_metrics, split="cv",
            params={"weights": WEIGHTS, "allocation": "vol_budget",
                    "target_vol_ratio": TARGET_VOL_RATIO,
                    "final_n_iter": N_FINAL, **BEST},
            notes="LightGBM + CatBoost (Plain boosting) + Ridge on standardised "
                  "predictions; weights tuned on fold-mean Spearman; sized with "
                  "vol_budget_allocation; early-stopped refit")
save_result(model=MODEL_NAME, stage="improved", metrics=public_metrics,
            split="public", params={"weights": WEIGHTS, "allocation": "vol_budget"},
            notes="refit on full train period with early-stopped iteration "
                  "counts, scored once on held-out 180 rows")

save_predictions(MODEL_NAME, OOF_ROWS[DATE_COL], Y_OOF, blended_oof,
                 FINAL_POS_OOF)

print("=== cross-validated (identical purged folds) ===")
display(compare(split="cv"))
print("=== held-out public block (180 rows, scored once) ===")
display(compare(split="public"))

=== cross-validated (identical purged folds) ===


,model,stage,split,spearman_ic_mean,spearman_ic,rmse_mean,rmse,modified_sharpe_mean,modified_sharpe,sharpe_mean,sharpe,vol_ratio_mean,vol_ratio
0,improved_ensemble,improved,cv,0.078414,NaN,0.993576,NaN,0.762293,NaN,0.801426,NaN,1.274340,NaN
1,hybrid_domain_lgbm,proposed,cv,0.075462,NaN,0.010853,NaN,0.705666,NaN,0.759851,NaN,0.745949,NaN
2,improved_proposal,improved,cv,0.053835,NaN,0.990918,NaN,0.580561,NaN,0.578746,NaN,1.193926,NaN


=== held-out public block (180 rows, scored once) ===


,model,stage,split,spearman_ic_mean,spearman_ic,rmse_mean,rmse,modified_sharpe_mean,modified_sharpe,sharpe_mean,sharpe,vol_ratio_mean,vol_ratio
0,hybrid_domain_lgbm,proposed,public,NaN,0.180720,NaN,0.010549,NaN,1.868358,NaN,1.868358,NaN,0.803296
1,improved_proposal,improved,public,NaN,0.067866,NaN,0.937807,NaN,0.943326,NaN,0.949047,NaN,1.193106
2,improved_ensemble,improved,public,NaN,0.165618,NaN,1.447199,NaN,0.989735,NaN,0.993080,NaN,1.203868
